Load essentia tensorflow library to load and process audio files

In [ ]:
# Install Conda
# https://www.anaconda.com/docs/getting-started/anaconda/install#linux-installer
# conda create -n ess python=3.11
# conda activate ess
# conda install -c conda-forge -y cudatoolkit=11.2 cudnn=8.1
#!pip install essentia-tensorflow
#!pip install "numpy<2.0"

  Using cached essentia_tensorflow-2.1b6.dev1110-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.9 kB)
Using cached essentia_tensorflow-2.1b6.dev1110-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (291.4 MB)


Download the Discogs-Effnet model, the genre classifier and the genre classifier json.

In [3]:
#!wget https://essentia.upf.edu/models/music-style-classification/discogs-effnet/discogs-effnet-bs64-1.pb
#!wget https://essentia.upf.edu/models/classification-heads/mtg_jamendo_genre/mtg_jamendo_genre-discogs-effnet-1.pb
#!wget https://essentia.upf.edu/models/classification-heads/mtg_jamendo_genre/mtg_jamendo_genre-discogs-effnet-1.json


In [5]:
import essentia.standard as es
import numpy as np
import pandas as pd
import os
import json
from tqdm import tqdm

DEFAULT_SAMPLE_RATE = 16000
THRESHOLD_FOR_GENRE = 0.5

def load_genre_classes(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    return data["classes"]

def load_audio(audio_path, sample_rate=DEFAULT_SAMPLE_RATE):
    return es.MonoLoader(filename=audio_path, sampleRate=sample_rate, resampleQuality=4)()

def extract_embeddings(audio, embedding_model):
    return embedding_model(audio)

def predict_genres(embeddings, genre_classifier):
    return genre_classifier(embeddings)

def process_predictions(predictions, genres, threshold=THRESHOLD_FOR_GENRE, top_n=10):
    genre_scores = [(genres[i], float(predictions[0][i])) for i in range(len(genres))]
    predicted_genres = [genre for genre, score in genre_scores if score > threshold]
    sorted_scores = sorted(genre_scores, key=lambda x: x[1], reverse=True)
    top_genres = [genre for genre, score in sorted_scores[:top_n]]
    top_scores = [score for genre, score in sorted_scores[:top_n]]
    return predicted_genres, top_genres, top_scores

def process_dataset(dataset_dir, embedding_model, genre_classifier, genres, threshold=0.5, top_n=3):
    results = []
    for fname in tqdm(os.listdir(dataset_dir)):
        if fname.endswith(".wav") or fname.endswith(".mp3"):
            audio_path = os.path.join(dataset_dir, fname)
            audio = load_audio(audio_path)
            embeddings = extract_embeddings(audio, embedding_model)
            predictions = predict_genres(embeddings, genre_classifier)
            predicted_genres, top_genres, top_scores = process_predictions(
                predictions, genres, threshold, top_n
            )
            results.append({
                "filename": fname,
                "predicted_genres": predicted_genres,
                f"top_{top_n}_genres": top_genres,
                "scores": top_scores
            })
    return pd.DataFrame(results)

In [6]:
dataset_dir = "../datasets/mvsep_multisong_dataset"
embedding_model = es.TensorflowPredictEffnetDiscogs(
    graphFilename="discogs-effnet-bs64-1.pb",
    output="PartitionedCall:1"
)
genre_classifier = es.TensorflowPredict2D(
    graphFilename="mtg_jamendo_genre-discogs-effnet-1.pb",
    output="model/Sigmoid"
)
genres = load_genre_classes("mtg_jamendo_genre-discogs-effnet-1.json")

2025-04-20 03:46:26.940509: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-04-20 03:46:26.947571: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 3060 Laptop GPU computeCapability: 8.6
coreClock: 1.702GHz coreCount: 30 deviceMemorySize: 5.68GiB deviceMemoryBandwidth: 312.97GiB/s
2025-04-20 03:46:26.947605: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1766] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-04-20 03:46:26.947618: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1258] D

In [ ]:
df = process_dataset(dataset_dir, embedding_model, genre_classifier, genres)
df.to_csv("mvsep_genre_predictions.csv", index=False)
df.head()